# Topic Modeling in Google Colab

This notebook recreates the topic-model workflow from the local Python script, but is set up to run in Google Colab.

Use the setup block to install packages, then choose one data source below: either upload a CSV, mount a Google Drive folder, or pull a Kaggle dataset via `kagglehub`.


In [1]:
%pip install -q pandas numpy matplotlib gensim nltk scikit-learn kagglehub

print('Packages installed.')


Note: you may need to restart the kernel to use updated packages.
Packages installed.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Choose a dataset source

If your CSV is in Google Drive, use the drive option. If it is on Kaggle, the Kaggle option downloads it automatically. If you want to upload from your machine, use the upload option.

For other datasets, a package-based import is also often the easiest option, for example: `kagglehub` for Kaggle or `datasets` for Hugging Face.


In [ ]:
import os
from pathlib import Path

DATA_SOURCE = 'kaggle'  # options: 'drive', 'kaggle', 'upload'
TEXT_COLUMN = 'headline_text'  # change this for other datasets, e.g. 'abstract'
DATA_PATH = '/content/drive/MyDrive/CUNY/AI_ML_2026/abcnews-date-text.csv'

if DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f'File not found in Drive: {DATA_PATH}. Update DATA_PATH to match your file path.')

elif DATA_SOURCE == 'kaggle':
    import kagglehub
    dataset_path = kagglehub.dataset_download('therohk/million-headlines')
    DATA_PATH = os.path.join(dataset_path, 'abcnews-date-text.csv')
    print('Kaggle dataset downloaded to:', dataset_path)

elif DATA_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = next(iter(uploaded))


print('Data source selected:', DATA_SOURCE)
print('Data path:', DATA_PATH)
print('Exists:', os.path.exists(DATA_PATH))

# Optional example for Hugging Face datasets:
# from datasets import load_dataset
# ds = load_dataset('CShorten/ML-ArXiv-Papers')
# print(ds)


Data source selected: data\wikidata_country_numerical.csv
Data path: data\wikidata_country_numerical.csv
Exists: False


In [15]:
import os
from pathlib import Path

TEXT_COLUMN = 'countryLabel'
DATA_PATH = Path("../../data") / "wikidata_country_numerical.csv"
print('Exists:', DATA_PATH.exists())

Exists: True


Basic Imports

In [16]:
import numpy as np
import pandas as pd
import gensim
from gensim import corpora
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models.coherencemodel import CoherenceModel
from nltk.stem import WordNetLemmatizer, SnowballStemmer
import nltk
import matplotlib.pyplot as plt


Settings

In [17]:

np.random.seed(2026)
nltk.download('wordnet')
nltk.download('omw-1.4')
stemmer = SnowballStemmer('english')

# model settings
NUM_TOPICS = 25
PASSES = 10
WORKERS = 1

# data settings
USE_SUBSET = True
DO_TESTING = True
TRUNCATE_SIZE = 50000
INSPECT_ROW = 4310

# dictionary settings
MAX_WORDS = 100000
MIN_WORDS = 15
MAX_PERCENTAGE = 0.5

print('Dependencies loaded.')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\richa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\richa\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Dependencies loaded.


Preprocessing functions

In [18]:
def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

def preprocess(text):
    if pd.isna(text) or not isinstance(text, str):
        return []

    result = []
    for token in gensim.utils.simple_preprocess(text):
        if token not in STOPWORDS and len(token) > 3:
            result.append(lemmatize_stemming(token))
    return result

print('Preprocessing functions ready.')


Preprocessing functions ready.


In [19]:
# Load data from CSV
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'Could not find the dataset at {DATA_PATH}. Check the file path or change DATA_SOURCE.')

data = pd.read_csv(DATA_PATH)
print('Dataset loaded. Shape:', data.shape)
print(data.head())
print('Missing values:\n', data.isna().sum())

if TEXT_COLUMN not in data.columns:
    raise KeyError(f'Text column {TEXT_COLUMN} not found. Available columns: {list(data.columns[:10])}')

data_text = data[[TEXT_COLUMN]].copy()
data_text['index'] = data_text.index

documents = data_text
if USE_SUBSET:
    documents = data_text.truncate(before=0, after=TRUNCATE_SIZE)

print('Documents kept for analysis:', len(documents))
print(documents.head())


Dataset loaded. Shape: (200, 6)
                               country countryLabel         pop    areaKm2  \
0  http://www.wikidata.org/entity/Q889  Afghanistan  41454761.0   652230.0   
1  http://www.wikidata.org/entity/Q222      Albania   2811655.0    28748.0   
2  http://www.wikidata.org/entity/Q262      Algeria  46164219.0  2381741.0   
3  http://www.wikidata.org/entity/Q228      Andorra     89752.0      468.0   
4  http://www.wikidata.org/entity/Q916       Angola  36749906.0  1246700.0   

         gdpUSD  lifeExp  
0  2.049713e+10   63.673  
1  2.354718e+10   78.345  
2  1.919129e+11   77.130  
3  3.352033e+09      NaN  
4  1.359668e+11   61.547  
Missing values:
 country          0
countryLabel     0
pop              1
areaKm2          1
gdpUSD          14
lifeExp         10
dtype: int64
Documents kept for analysis: 200
  countryLabel  index
0  Afghanistan      0
1      Albania      1
2      Algeria      2
3      Andorra      3
4       Angola      4


In [10]:
# Inspect a sample row before and after preprocessing
if DO_TESTING:
    print('\nDocument length is', len(documents))
    print(documents.head())

    doc_sample = documents[documents['index'] == INSPECT_ROW].values[0][0]
    print(f'\nOriginal row #{INSPECT_ROW}:')
    print(doc_sample)
    print(f'\nTokenized and lemmatized row #{INSPECT_ROW}:')
    print(preprocess(doc_sample))



Document length is 18819
  abstract  index
0      NaN      0
1      NaN      1
2      NaN      2
3      NaN      3
4      NaN      4

Original row #4310:
The exploration of the theme and narration of silence in Virginia Woolf's novels in this dissertation brings under scrutiny nothing less than her perceptions of the nature of gender, being, mind, knowledge and language. Woolf, searching for a language of mind amid changing concepts of mind in the twentieth century, creates a new rhetoric of silence. In defining silence as a "presence" and not just an "absence" in life and narration, Woolf displaces the privileged place of the "speaking subject" and speech or dialogue in the novel. In Chapter 1, there is an attempt to define what "silence" is in a text. It is about a theory of reading silence in any text as well as a reading of silence in Virginia Woolf's novels. Chapter 2, The Reading of Silence, discusses Woolf's notions of the reader and the writer, and how the critical perspective

In [ ]:
# Local fallback; Colab users can replace this with a Drive or uploaded file path.
DATA_PATH = Path("../data/raw/gc_dissertations_combined_v2.csv")


First 10 processed documents:
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]

First 10 dictionary entries:
0 abstract
1 provid
2 avail
3 addit
4 adequ
5 analys
6 analysi
7 analyz
8 approxim
9 associ
Dictionary size after filtering: 7411
Bag-of-words corpus size: 18819

Example BOW output for row #4310:
Word ID: 1 (provid) appears 2 time(s).
Word ID: 42 (indic) appears 1 time(s).
Word ID: 93 (absenc) appears 1 time(s).
Word ID: 117 (discuss) appears 1 time(s).
Word ID: 125 (experiment) appears 1 time(s).
Word ID: 156 (methodolog) appears 1 time(s).
Word ID: 171 (present) appears 1 time(s).
Word ID: 188 (select) appears 2 time(s).
Word ID: 195 (subject) appears 1 time(s).
Word ID: 198 (theori) appears 1 time(s).


In [12]:
# Optional coherence sweep: test multiple topic counts
DO_COHERENCE_TEST = False

if DO_COHERENCE_TEST:
    def compute_coherence_values(dictionary, corpus, texts, start=2, limit=40, step=6):
        coherence_values = []
        for num_topics in range(start, limit, step):
            lda_model = gensim.models.LdaMulticore(
                corpus,
                num_topics=num_topics,
                id2word=dictionary,
                passes=10,
                workers=1,
            )
            cm = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
            coherence_values.append(cm.get_coherence())
        return coherence_values

    coherence_values = compute_coherence_values(
        dictionary=dictionary,
        corpus=bow_corpus,
        texts=processed_docs,
        start=10,
        limit=200,
        step=15,
    )

    x = list(range(10, 200, 15))
    plt.plot(x, coherence_values)
    plt.xlabel('Num Topics')
    plt.ylabel('Coherence score')
    plt.title('Topic Coherence by Number of Topics')
    plt.show()
else:
    print('Coherence sweep skipped. Set DO_COHERENCE_TEST = True to run it.')


Coherence sweep skipped. Set DO_COHERENCE_TEST = True to run it.


In [13]:
# Train the LDA model
try:
    print('Creating LDA model...')
    lda_model = gensim.models.LdaMulticore(
        bow_corpus,
        num_topics=NUM_TOPICS,
        id2word=dictionary,
        passes=PASSES,
        workers=WORKERS,
    )

    print('Model created. Top terms by topic:')
    for idx, topic in lda_model.print_topics(-1):
        print(f'Topic: {idx} | Words: {topic}')
except Exception as e:
    print('Something went wrong while training the model.')
    print(str(e))


Creating LDA model...
Model created. Top terms by topic:
Topic: 0 | Words: 0.029*"polit" + 0.018*"black" + 0.016*"american" + 0.014*"social" + 0.013*"ident" + 0.013*"racial" + 0.011*"cultur" + 0.011*"nation" + 0.010*"movement" + 0.009*"immigr"
Topic: 1 | Words: 0.023*"cultur" + 0.019*"chapter" + 0.013*"dissert" + 0.010*"context" + 0.009*"analysi" + 0.008*"explor" + 0.008*"practic" + 0.007*"voic" + 0.007*"tradit" + 0.007*"understand"
Topic: 2 | Words: 0.085*"dollar" + 0.034*"bind" + 0.030*"protein" + 0.017*"structur" + 0.016*"lcub" + 0.016*"rcub" + 0.012*"interact" + 0.011*"domain" + 0.010*"sequenc" + 0.010*"peptid"
Topic: 3 | Words: 0.037*"speci" + 0.016*"morpholog" + 0.014*"chapter" + 0.013*"host" + 0.012*"color" + 0.010*"evolutionari" + 0.010*"parasit" + 0.009*"relationship" + 0.009*"evolut" + 0.009*"group"
Topic: 4 | Words: 0.023*"social" + 0.019*"communiti" + 0.018*"citi" + 0.017*"work" + 0.014*"york" + 0.012*"research" + 0.012*"program" + 0.009*"public" + 0.009*"servic" + 0.008*"p

In [14]:
# Assign topics to unseen text
unseen_text = 'The stock market is experiencing unprecedented volatility due to global economic uncertainty.'

bow_vector = dictionary.doc2bow(preprocess(unseen_text))
print(f'\nUnseen text: {unseen_text}')
print('Top predicted topics:')
for index, score in sorted(lda_model[bow_vector], key=lambda tup: -tup[1]):
    print(f'Score: {score}\t Topic {index}: {lda_model.print_topic(index, 5)}')



Unseen text: The stock market is experiencing unprecedented volatility due to global economic uncertainty.
Top predicted topics:
Score: 0.4611439108848572	 Topic 12: 0.020*"chapter" + 0.017*"model" + 0.015*"market" + 0.013*"firm" + 0.012*"effect"
Score: 0.43661603331565857	 Topic 17: 0.019*"econom" + 0.013*"state" + 0.013*"develop" + 0.012*"labor" + 0.011*"industri"


## Notes and next steps

- Change `TEXT_COLUMN` to match your dataset if you are using an abstract or description field rather than a headline column.
- For large datasets, start with a subset of rows using `USE_SUBSET = True` and `TRUNCATE_SIZE`.
- The model is interpretive: topic labels depend heavily on preprocessing choices, stopwords, and the number of topics.
- If you want to save the model or export topic assignments: use `lda_model.save(...)` and merge the topic scores back into your DataFrame.
- Next possible improvements: tune the number of topics using coherence scoring, add custom stopwords, and compare lemmatization vs. stemming.


## Sidebar: find your Google drive path

If you have any trouble, you can navigate via ls and chdir

In [ ]:
# mount the drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# understand where you are
os.listdir()

['abcnews-date-text.csv', 'AI and ML Syllabus.gdoc']

In [ ]:
# move somewhere else and figure out what is there
os.chdir("/content/drive/MyDrive/")
os.listdir()


['abcnews-date-text.csv', 'AI and ML Syllabus.gdoc']

### Drive issues

If you are trying to use Google Drive to host your files, the initial connection can be tricky. If you get an authentication erro, you may need to adjust your account permissions.

To check and manage your Google Account permissions for third-party apps. Here is some Gemini copypasta that worked when I tested it:

* Go to your Google Account: Open your web browser and go to [redacted link].
* Navigate to Security: On the left-hand
navigation panel, click on 'Security'.
* Find 'Third-party apps with account access': Scroll down to the section titled 'Third-party apps with account access' or similar. It might be under 'Signing in to other sites' or 'Manage third-party access'.
* Manage Third-party Access: Click on 'Manage third-party access'.
* Review Connected Apps: You will see a list of applications that have access to your Google Account. Look for 'Google Colaboratory' or any similar entry. If you see it, ensure it has the necessary permissions. If you previously removed its access, you might need to re-authorize it when you try to mount Drive again in Colab.
* Remove Old/Unused Access: While you're there, it's good practice to remove access for any apps you no longer use or don't recognize. This helps maintain your account's security.
* After reviewing these settings, try running the drive.mount() cell in Colab again. If you removed and then re-granted access, or if you previously denied the Colab pop-up, a new authentication prompt should appear, which you'll need to accept.